# **CLIP Zero-Shot Classification**
## **Car Make & Model (Stanford Cars Dataset)**

---

In this notebook we use **OpenAI's CLIP** model in a **zero-shot** setting to predict the make and model of a car.

Zero-shot means the model has never been fine-tuned on car images – it uses its rich image-text representations to match a car photo against 196 text descriptions like `"a photo of a BMW 3 Series 2012"`. No training required!

We then measure how well zero-shot CLIP performs on the Stanford Cars test set compared to the fine-tuned ViT in `transfer_car_make_model_prediction.ipynb`.

In [ ]:
!pip install transformers datasets scikit-learn tqdm --quiet

In [ ]:
import torch
import io
import numpy as np
from tqdm import tqdm
from PIL import Image as PILImage
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

### Load CLIP model
---

In [ ]:
CLIP_MODEL_NAME = "openai/clip-vit-large-patch14"

model     = CLIPModel.from_pretrained(CLIP_MODEL_NAME)
processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)
print(f"Model loaded on {device}")

### Load dataset and build text prompts
---
For each of the 196 classes we create a natural-language prompt:
> `"a photo of a <make and model>"`

CLIP then scores each image against all 196 prompts and picks the highest.

In [ ]:
# --- Kaggle path ---
DATA_DIR = "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder"

raw_dataset = load_dataset("imagefolder", data_dir=DATA_DIR)
label_names = raw_dataset['train'].features['label'].names

print(f"Classes: {len(label_names)}")
print("Sample class names:", label_names[:5])

In [ ]:
# Build zero-shot text prompts for every class
TEXT_PROMPTS = [f"a photo of a {name}" for name in label_names]

print("Example prompts:")
for p in TEXT_PROMPTS[:5]:
    print(" ", p)

### Pre-compute text embeddings
---
We encode all 196 prompts once upfront to speed up inference. This is the standard CLIP zero-shot trick.

In [ ]:
# Encode all text prompts in one pass
text_inputs = processor(
    text=TEXT_PROMPTS,
    return_tensors="pt",
    padding=True,
    truncation=True,
).to(device)

with torch.no_grad():
    text_out = model.get_text_features(**text_inputs)  # may return tensor or object
    # unwrap if the model returns a dataclass/object instead of a raw tensor
    text_features = text_out if isinstance(text_out, torch.Tensor) else text_out.pooler_output
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)  # normalise

print("Text features shape:", text_features.shape)

### Run zero-shot classification on the test set
---
For efficiency we use a small subset for quick evaluation. Remove the `SAMPLE_SIZE` limit to evaluate the full test set (takes longer).

In [ ]:
# --- Limit to first N samples for quick evaluation (remove for full eval) ---
SAMPLE_SIZE = 200  # set to None to evaluate the entire test set

test_ds = raw_dataset['test']
if SAMPLE_SIZE:
    test_ds = test_ds.select(range(min(SAMPLE_SIZE, len(test_ds))))

print(f"Evaluating on {len(test_ds)} samples")

# Make sure images are not pre-decoded
from datasets import Image as HFImage
test_ds = test_ds.cast_column('image', HFImage(decode=False))

In [ ]:
true_labels      = []
predicted_labels = []
predicted_top5   = []

for i in tqdm(range(len(test_ds))):
    sample     = test_ds[i]
    image_data = sample['image']

    # Load image
    if isinstance(image_data, dict):
        path = image_data.get('path')
        if path:
            image = PILImage.open(path).convert('RGB')
        elif image_data.get('bytes'):
            image = PILImage.open(io.BytesIO(image_data['bytes'])).convert('RGB')
        else:
            continue
    else:
        continue

    # Encode image
    image_inputs  = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        image_out = model.get_image_features(**image_inputs)
        image_features = image_out if isinstance(image_out, torch.Tensor) else image_out.pooler_output
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    # Cosine similarity → softmax probabilities
    logits = (image_features @ text_features.T) * model.logit_scale.exp()
    probs  = logits.softmax(dim=-1)[0].cpu()

    pred_idx  = probs.argmax().item()
    top5_idxs = probs.topk(5).indices.tolist()

    true_labels.append(sample['label'])
    predicted_labels.append(pred_idx)
    predicted_top5.append(top5_idxs)

print("Done.")

### Metrics
---

In [ ]:
from sklearn.metrics import accuracy_score, top_k_accuracy_score
import numpy as np

accuracy   = accuracy_score(true_labels, predicted_labels)
top5_acc   = sum(
    true_labels[i] in predicted_top5[i] for i in range(len(true_labels))
) / len(true_labels)

print(f"Top-1 Accuracy : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"Top-5 Accuracy : {top5_acc:.4f}  ({top5_acc*100:.2f}%)")

### Visualise predictions
---

In [ ]:
import matplotlib.pyplot as plt

# Reload a decoded version for display
display_ds = raw_dataset['test']
if SAMPLE_SIZE:
    display_ds = display_ds.select(range(min(SAMPLE_SIZE, len(display_ds))))

ROWS, COLS = 3, 4
indices    = np.random.choice(len(display_ds), ROWS * COLS, replace=False)

fig = plt.figure(figsize=(COLS * 5, ROWS * 5))
for plot_i, idx in enumerate(indices):
    img        = display_ds[int(idx)]['image']
    true_lbl   = label_names[true_labels[int(idx)]]
    pred_lbl   = label_names[predicted_labels[int(idx)]]
    color      = 'green' if true_lbl == pred_lbl else 'red'

    ax = fig.add_subplot(ROWS, COLS, plot_i + 1)
    plt.imshow(img)
    plt.title(f"True: {true_lbl}\nPred: {pred_lbl}", fontsize=7, color=color)
    plt.axis('off')

plt.tight_layout()
plt.show()

### Gradio demo
---

In [ ]:
!pip install gradio --quiet

In [ ]:
import gradio as gr

def classify_car_zero_shot(image_path: str) -> dict:
    image         = PILImage.open(image_path).convert('RGB')
    image_inputs  = processor(images=image, return_tensors='pt').to(device)

    with torch.no_grad():
        image_out = model.get_image_features(**image_inputs)
        image_features = image_out if isinstance(image_out, torch.Tensor) else image_out.pooler_output
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

    logits = (image_features @ text_features.T) * model.logit_scale.exp()
    probs  = logits.softmax(dim=-1)[0].cpu()

    top5_values, top5_idxs = probs.topk(5)
    return {label_names[i.item()]: v.item() for i, v in zip(top5_idxs, top5_values)}

iface = gr.Interface(
    fn=classify_car_zero_shot,
    inputs=gr.Image(type='filepath'),
    outputs=gr.Label(num_top_classes=5),
    title='Car Make & Model – CLIP Zero-Shot',
    description='Upload a car photo. CLIP matches it against 196 class descriptions without any task-specific training.',
)

iface.launch()